# Análise de Pré-processamento — SINASC 2024

## Objetivo

Investigar os valores ausentes, códigos ignorados,
categorias e inconsistências dos atributos selecionados,
orientando a construção do pipeline de pré-processamento.

In [1]:
import pandas as pd
import numpy as np
import sys

from pathlib import Path

# Localizar a raiz do projeto
raiz = Path.cwd()

while not (raiz / ".git").exists():
    if raiz == raiz.parent:
        raise FileNotFoundError("Raiz do projeto não encontrada.")
    raiz = raiz.parent

# Permitir a importação dos scripts da pasta src
sys.path.insert(0, str(raiz / "ml" / "src"))

from data_preparation import (
    carregar_dados,
    criar_variavel_alvo,
    separar_atributos_alvo,
    COLUNAS_CATEGORICAS,
    COLUNAS_NUMERICAS
)

# Carregar e preparar os dados
df = carregar_dados()
dados = criar_variavel_alvo(df)

X, y = separar_atributos_alvo(dados)

print("Dados carregados com sucesso!")
print("Dimensões de X:", X.shape)
print("Dimensões de y:", y.shape)

Dados carregados com sucesso!
Dimensões de X: (120785, 10)
Dimensões de y: (120785,)


In [2]:
# investigar as categorias
for coluna in COLUNAS_CATEGORICAS:

    print(f"\nAtributo: {coluna}")
    print("-" * 40)

    print(
        X[coluna]
        .value_counts(dropna=False)
        .to_string()
    )


Atributo: FxEtPartur
----------------------------------------
FxEtPartur
25–29    30223
30–34    27908
20–24    24817
35–39    21788
15–19     8272
40–44     6973
45–49      509
10–14      249
>=50        45
Ign          1

Atributo: ESTCIVMAE
----------------------------------------
ESTCIVMAE
1.0    56395
2.0    41805
5.0    20260
4.0     2106
3.0      129
NaN       75
9.0       15

Atributo: ESCMAE2010
----------------------------------------
ESCMAE2010
3.0    66428
5.0    35491
2.0    10996
4.0     7031
1.0      762
0.0       45
9.0       24
NaN        8

Atributo: GRAVIDEZ
----------------------------------------
GRAVIDEZ
1.0    117543
2.0      3178
3.0        53
NaN         7
9.0         4


## Análise do acompanhamento pré-natal

Investigação dos valores ausentes e inválidos do
atributo MESPRENAT e sua relação com CONSPRENAT,
buscando diferenciar ausência de acompanhamento
pré-natal de informação desconhecida.

In [3]:
print("Distribuição de MESPRENAT:")

print(
    dados["MESPRENAT"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

print("\nQuantidade de valores ausentes:")
print(dados["MESPRENAT"].isna().sum())

print("\nQuantidade de registros com valor 99:")
print((dados["MESPRENAT"] == 99).sum())

print("\nQuantidade de registros com valor 10:")
print((dados["MESPRENAT"] == 10).sum())

Distribuição de MESPRENAT:
MESPRENAT
1.0     36402
2.0     55924
3.0     15312
4.0      5500
5.0      3081
6.0      1788
7.0       951
8.0       598
9.0       206
10.0        2
99.0      110
NaN       911

Quantidade de valores ausentes:
911

Quantidade de registros com valor 99:
110

Quantidade de registros com valor 10:
2


In [4]:
# Selecionar registros com MESPRENAT ausente
sem_mes_prenatal = dados[
    dados["MESPRENAT"].isna()
]

print("Total de registros com MESPRENAT ausente:")
print(len(sem_mes_prenatal))

print("\nQuantidade de consultas nesses registros:")

print(
    sem_mes_prenatal["CONSPRENAT"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

Total de registros com MESPRENAT ausente:
911

Quantidade de consultas nesses registros:
CONSPRENAT
0.0     857
2.0       1
3.0       1
4.0       1
5.0       2
6.0       6
7.0       3
8.0       2
9.0       3
10.0      1
11.0      1
12.0      3
13.0      1
14.0      2
99.0      1
NaN      26


## Análise dos atributos numéricos

Investigação das distribuições, valores ausentes,
valores extremos e possíveis inconsistências dos
atributos numéricos selecionados para o modelo.

Os resultados orientarão as regras de tratamento
e qualidade dos dados.

In [5]:
display(
    X[COLUNAS_NUMERICAS]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
QTDGESTANT,120766.0,1.166951,1.358364,0.0,0.0,1.0,2.0,24.0
QTDPARTNOR,120762.0,0.584787,1.038284,0.0,0.0,0.0,1.0,36.0
QTDPARTCES,120767.0,0.315467,0.639415,0.0,0.0,0.0,0.0,30.0
QTDFILVIVO,120761.0,0.884855,1.116700,0.0,0.0,1.0,1.0,11.0
QTDFILMORT,120747.0,0.288562,0.649232,0.0,0.0,0.0,0.0,26.0
MESPRENAT,119874.0,2.223368,3.183122,1.0,1.0,2.0,2.0,99.0


In [6]:
for coluna in COLUNAS_NUMERICAS:

    print(f"\nAtributo: {coluna}")
    print("-" * 40)

    print("Valores ausentes:", X[coluna].isna().sum())

    print("Valores negativos:", (X[coluna] < 0).sum())

    print("Valores iguais a 99:", (X[coluna] == 99).sum())

    print("Valor mínimo:", X[coluna].min())

    print("Valor máximo:", X[coluna].max())


Atributo: QTDGESTANT
----------------------------------------
Valores ausentes: 19
Valores negativos: 0
Valores iguais a 99: 0
Valor mínimo: 0.0
Valor máximo: 24.0

Atributo: QTDPARTNOR
----------------------------------------
Valores ausentes: 23
Valores negativos: 0
Valores iguais a 99: 0
Valor mínimo: 0.0
Valor máximo: 36.0

Atributo: QTDPARTCES
----------------------------------------
Valores ausentes: 18
Valores negativos: 0
Valores iguais a 99: 0
Valor mínimo: 0.0
Valor máximo: 30.0

Atributo: QTDFILVIVO
----------------------------------------
Valores ausentes: 24
Valores negativos: 0
Valores iguais a 99: 0
Valor mínimo: 0.0
Valor máximo: 11.0

Atributo: QTDFILMORT
----------------------------------------
Valores ausentes: 38
Valores negativos: 0
Valores iguais a 99: 0
Valor mínimo: 0.0
Valor máximo: 26.0

Atributo: MESPRENAT
----------------------------------------
Valores ausentes: 911
Valores negativos: 0
Valores iguais a 99: 110
Valor mínimo: 1.0
Valor máximo: 99.0


In [7]:
# Valores distintos
for coluna in COLUNAS_NUMERICAS:

    print(f"\nAtributo: {coluna}")
    print("-" * 40)

    print(
        X[coluna]
        .value_counts()
        .sort_index(ascending=False)
        .head(10)
        .to_string()
    )


Atributo: QTDGESTANT
----------------------------------------
QTDGESTANT
24.0     1
23.0     1
20.0     1
16.0     1
15.0     1
14.0     2
13.0     4
12.0     6
11.0    13
10.0    45

Atributo: QTDPARTNOR
----------------------------------------
QTDPARTNOR
36.0      1
30.0      1
20.0      2
11.0      2
10.0     11
9.0      22
8.0      55
7.0     109
6.0     268
5.0     655

Atributo: QTDPARTCES
----------------------------------------
QTDPARTCES
30.0       2
22.0       1
20.0       1
10.0       2
8.0        1
7.0        1
6.0        6
5.0       35
4.0      204
3.0     1086

Atributo: QTDFILVIVO
----------------------------------------
QTDFILVIVO
11.0        3
10.0       10
9.0        29
8.0        73
7.0       146
6.0       315
5.0       885
4.0      2288
3.0      6164
2.0     16579

Atributo: QTDFILMORT
----------------------------------------
QTDFILMORT
26.0     1
20.0     2
18.0     1
15.0     1
12.0     1
11.0     2
10.0     5
9.0      3
8.0     14
7.0     14

Atributo: MESPRENAT

## Análise de consistência do histórico gestacional

Verificação de possíveis inconsistências entre o
número de gestações anteriores e o número de partos
registrados.

Os resultados serão utilizados para orientar as
regras de qualidade dos dados em conjunto com
a Governança de Dados.

In [8]:
# Soma dos partos vaginais e cesáreos
total_partos = (
    X["QTDPARTNOR"] + X["QTDPARTCES"]
)

# Identificar possíveis inconsistências
inconsistencias = pd.Series({
    "Partos vaginais > gestações anteriores":
        (X["QTDPARTNOR"] > X["QTDGESTANT"]).sum(),

    "Partos cesáreos > gestações anteriores":
        (X["QTDPARTCES"] > X["QTDGESTANT"]).sum(),

    "Total de partos > gestações anteriores":
        (total_partos > X["QTDGESTANT"]).sum(),

    "Partos vaginais acima de 15":
        (X["QTDPARTNOR"] > 15).sum(),

    "Partos cesáreos acima de 15":
        (X["QTDPARTCES"] > 15).sum()
})

display(inconsistencias.to_frame("Quantidade"))

,Quantidade
Partos vaginais > gestações anteriores,70
Partos cesáreos > gestações anteriores,45
Total de partos > gestações anteriores,176
Partos vaginais acima de 15,4
Partos cesáreos acima de 15,4


## Validação da limpeza dos atributos

Aplicação da função de pré-processamento para verificar
o tratamento dos códigos inválidos, a padronização
das categorias e a criação do indicador de ausência
da informação de início do pré-natal.

In [9]:
from preprocessing import limpar_atributos

# Aplicar a limpeza
X_limpo = limpar_atributos(X)

print("Dimensões antes da limpeza:", X.shape)
print("Dimensões depois da limpeza:", X_limpo.shape)

print("\nAtributos após a limpeza:")

for coluna in X_limpo.columns:
    print("-", coluna)

Dimensões antes da limpeza: (120785, 10)
Dimensões depois da limpeza: (120785, 11)

Atributos após a limpeza:
- FxEtPartur
- ESTCIVMAE
- ESCMAE2010
- GRAVIDEZ
- QTDGESTANT
- QTDPARTNOR
- QTDPARTCES
- QTDFILVIVO
- QTDFILMORT
- MESPRENAT
- MESPRENAT_AUSENTE


In [10]:
# comparação de valores ausentes
# Essa tabela mostra quantos valores inválidos foram convertidos em valores ausentes.
comparacao = pd.DataFrame({
    "Antes": X.isna().sum(),
    "Depois": X_limpo[X.columns].isna().sum()
})

comparacao["Diferenca"] = (
    comparacao["Depois"] - comparacao["Antes"]
)

display(comparacao)

,Antes,Depois,Diferenca
FxEtPartur,0,1,1
ESTCIVMAE,75,90,15
ESCMAE2010,8,32,24
GRAVIDEZ,7,11,4
QTDGESTANT,19,19,0
QTDPARTNOR,23,23,0
QTDPARTCES,18,18,0
QTDFILVIVO,24,24,0
QTDFILMORT,38,38,0
MESPRENAT,911,1023,112


In [11]:
# 0 — Informação válida
# 1 — Informação ausente ou inválida
indicador = X_limpo["MESPRENAT_AUSENTE"]

display(
    pd.DataFrame({
        "Quantidade": indicador.value_counts().sort_index(),
        "Percentual (%)": (
            indicador.value_counts(normalize=True)
            .sort_index() * 100
        ).round(2)
    })
)

,Quantidade,Percentual (%)
MESPRENAT_AUSENTE,,
0,119762,99.15
1,1023,0.85


## Separação dos conjuntos de treinamento e teste

A base será dividida em:

- 80% dos registros para treinamento.
- 20% dos registros para teste.

A divisão será estratificada para preservar a proporção
das classes da variável-alvo. O conjunto de teste será reservado para a avaliação final do modelo, evitando vazamento de informações durante o treinamento e a seleção dos modelos.

In [12]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Dimensões de X_treino:", X_treino.shape)
print("Dimensões de X_teste:", X_teste.shape)

print("Dimensões de y_treino:", y_treino.shape)
print("Dimensões de y_teste:", y_teste.shape)

Dimensões de X_treino: (96628, 10)
Dimensões de X_teste: (24157, 10)
Dimensões de y_treino: (96628,)
Dimensões de y_teste: (24157,)


In [13]:
distribuicao_classes = pd.DataFrame({
    "Base original (%)":
        y.value_counts(normalize=True) * 100,

    "Treinamento (%)":
        y_treino.value_counts(normalize=True) * 100,

    "Teste (%)":
        y_teste.value_counts(normalize=True) * 100
})

display(distribuicao_classes.sort_index().round(2))

,Base original (%),Treinamento (%),Teste (%)
baixo_peso,,,
0,89.67,89.67,89.68
1,10.33,10.33,10.32


nossa base apresenta desbalanceamento significativo. A estratificação garante que os conjuntos de treinamento e teste tenham aproximadamente a mesma proporção de casos de baixo peso.

## Validação do Pipeline completo

Teste inicial utilizando o DummyClassifier para validar
a integração entre a limpeza dos dados, a imputação,
a padronização, a codificação das categorias e o
classificador.

O teste será realizado exclusivamente com os dados
de treinamento, preservando o conjunto de teste
para a avaliação final.

In [14]:
from sklearn.dummy import DummyClassifier
from preprocessing import criar_pipeline

# Criar o classificador de referência
modelo_dummy = DummyClassifier(
    strategy="most_frequent"
)

# Construir o Pipeline completo
pipeline_dummy = criar_pipeline(modelo_dummy)

# Ajustar o Pipeline aos dados de treinamento
pipeline_dummy.fit(X_treino, y_treino)

print("Pipeline treinado com sucesso!")

Pipeline treinado com sucesso!


## Validação dos dados transformados

Verificação das dimensões dos dados após o
pré-processamento e confirmação de que não existem
valores ausentes após a imputação.

In [15]:
# Aplicar a limpeza e o pré-processamento
X_transformado = pipeline_dummy[:-1].transform(X_treino)

print("Dimensões originais:", X_treino.shape)
print("Dimensões após transformação:", X_transformado.shape)

# Converter para array, caso seja uma matriz esparsa
if hasattr(X_transformado, "toarray"):
    X_transformado = X_transformado.toarray()

# Verificar valores ausentes e infinitos
print("Valores ausentes:", np.isnan(X_transformado).sum())
print("Valores infinitos:", np.isinf(X_transformado).sum())

Dimensões originais: (96628, 10)
Dimensões após transformação: (96628, 34)
Valores ausentes: 0
Valores infinitos: 0


## Validação das predições do Pipeline

Verificação da capacidade do pipeline de receber
os atributos no formato original e realizar
predições automaticamente.

O teste utiliza uma pequena amostra do conjunto
de treinamento, preservando o conjunto de teste
para a avaliação final.

In [16]:
# Selecionar cinco registros de treinamento
amostra = X_treino.head(5)

# Realizar predições
predicoes = pipeline_dummy.predict(amostra)

# Obter as probabilidades das classes
probabilidades = pipeline_dummy.predict_proba(amostra)

print("Dimensões da entrada:", amostra.shape)

print("\nClasses previstas:")
print(predicoes)

print("\nProbabilidades das classes:")
print(probabilidades)

print("\nClasses reconhecidas pelo modelo:")
print(pipeline_dummy.classes_)

Dimensões da entrada: (5, 10)

Classes previstas:
[0 0 0 0 0]

Probabilidades das classes:
[[1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]]

Classes reconhecidas pelo modelo:
[0 1]


## Validação da imputação categórica

Verificação das categorias geradas pelo OneHotEncoder,
confirmando que as informações desconhecidas são
representadas pela categoria Ignorado.

In [17]:
preprocessador = pipeline_dummy.named_steps["preprocessamento"]

encoder = (
    preprocessador
    .named_transformers_["categorico"]
    .named_steps["codificacao"]
)

for coluna, categorias in zip(
    COLUNAS_CATEGORICAS,
    encoder.categories_
):
    print(f"\nAtributo: {coluna}")
    print(categorias)


Atributo: FxEtPartur
['10-14' '15-19' '20-24' '25-29' '30-34' '35-39' '40-44' '45-49' '>=50'
 'Ignorado']

Atributo: ESTCIVMAE
['1' '2' '3' '4' '5' 'Ignorado']

Atributo: ESCMAE2010
['0' '1' '2' '3' '4' '5' 'Ignorado']

Atributo: GRAVIDEZ
['1' '2' '3' 'Ignorado']


## Teste de compatibilidade das entradas da API

Verificação da padronização dos códigos categóricos
recebidos em diferentes formatos, garantindo consistência
entre os dados de treinamento e as futuras predições.

In [18]:
# exemplos fictícios para testar a integração
amostra_api = pd.DataFrame({
    "FxEtPartur": ["25–29", "25-29", "25-29"],
    "ESTCIVMAE": [1, 1.0, "1"],
    "ESCMAE2010": [3, 3.0, "3"],
    "GRAVIDEZ": [1, 1.0, "1"],
    "QTDGESTANT": [1, 1, 1],
    "QTDPARTNOR": [1, 1, 1],
    "QTDPARTCES": [0, 0, 0],
    "QTDFILVIVO": [1, 1, 1],
    "QTDFILMORT": [0, 0, 0],
    "MESPRENAT": [2, 2, 2]
})

amostra_limpa = limpar_atributos(amostra_api)

display(amostra_limpa[COLUNAS_CATEGORICAS])

# Verificar se os três registros ficaram equivalentes
assert len(amostra_limpa.drop_duplicates()) == 1

print("Teste de compatibilidade: OK")

,FxEtPartur,ESTCIVMAE,ESCMAE2010,GRAVIDEZ
0,25-29,1,3,1
1,25-29,1,3,1
2,25-29,1,3,1


Teste de compatibilidade: OK



## Conclusão do pré-processamento

A etapa de pré-processamento permitiu identificar e
tratar inconsistências nos atributos selecionados
para o modelo de classificação.

### Tratamentos realizados

- Padronização das faixas etárias utilizando hífen simples.
- Tratamento dos códigos categóricos inválidos.
- Padronização dos códigos categóricos como texto.
- Conversão dos valores inválidos em NaN.
- Validação dos atributos numéricos.
- Tratamento dos valores inválidos de MESPRENAT.
- Criação do indicador MESPRENAT_AUSENTE.
- Imputação dos atributos numéricos pela mediana.
- Imputação dos atributos categóricos com a categoria Ignorado.
- Padronização numérica com StandardScaler.
- Codificação categórica com OneHotEncoder.

### Resultados da limpeza

- 120.785 registros preservados.
- 10 atributos originais de entrada.
- 1 atributo derivado internamente.
- 156 valores inválidos convertidos em ausentes.
- 1.023 registros com MESPRENAT ausente ou inválido.

### Divisão dos dados

- Treinamento: 96.628 registros (80%).
- Teste: 24.157 registros (20%).
- Divisão estratificada.
- Random state: 42.

### Pipeline implementado

O pipeline completo utiliza FunctionTransformer,
ColumnTransformer e um classificador.

As transformações que dependem de parâmetros aprendidos
são ajustadas exclusivamente nos dados de treinamento,
evitando vazamento entre treinamento e teste.

### Testes realizados

- Validação da limpeza dos dados.
- Treinamento inicial com DummyClassifier.
- Verificação de valores ausentes e infinitos.
- Teste de predição com os atributos originais.
- Validação da categoria Ignorado.
- Teste de compatibilidade dos formatos de entrada.

### Limitações

O indicador MESPRENAT_AUSENTE representa a ausência
de uma informação válida sobre o início do pré-natal,
não necessariamente a ausência de acompanhamento.

Os valores extremos e as possíveis inconsistências
do histórico gestacional foram mantidos provisoriamente,
aguardando alinhamento com a Governança de Dados.

### Próximas etapas

- Comparação dos modelos de classificação.
- Validação cruzada estratificada.
- Tratamento do desbalanceamento.
- Seleção do melhor modelo.
- Avaliação e exportação do modelo definitivo.
